# Clasificadores para varios datasets

En este cuaderno se comparan varios clasificadores sobre cuatro bases de datos diferentes:

- MNIST Digits
- Fashion-MNIST
- Wine Dataset
- Breast Cancer Wisconsin

La idea es probar cómo se comportan modelos clásicos de clasificación bajo distintos tipos de datos: imágenes, atributos numéricos y datos tabulares con diferente número de clases.

## Modelos a comparar

- Regresión Logística
- K-NN
- SVM (Support Vector Machine)
- Árbol de Decisión
- Random Forest

## Objetivo

Evaluar la precisión de cada modelo y observar cómo cambia el rendimiento según la complejidad del conjunto de datos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import sys

from sklearn.datasets import load_digits, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

try:
    from tensorflow.keras.datasets import fashion_mnist
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow-cpu", "-q"])
        from tensorflow.keras.datasets import fashion_mnist
    except Exception:
        raise ImportError("No se pudo cargar Fashion-MNIST. Instala tensorflow-cpu o usa otro dataset.")

# -----------------------------
# 1. Cargar datasets
# -----------------------------

def cargar_fashion_mnist(max_per_class=500):
    (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
    X = np.concatenate([x_train, x_test], axis=0).reshape(-1, 784).astype(np.float32) / 255.0
    y = np.concatenate([y_train, y_test], axis=0).astype(int)

    idx = []
    for clase in range(10):
        indices = np.where(y == clase)[0]
        idx.extend(indices[:max_per_class])

    idx = np.array(idx, dtype=int)
    return X[idx], y[idx]

mnist = load_digits()
X_mnist, y_mnist = mnist.data, mnist.target

X_fashion, y_fashion = cargar_fashion_mnist(max_per_class=500)

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

breast = load_breast_cancer()
X_breast, y_breast = breast.data, breast.target

print("MNIST Digits:", X_mnist.shape, y_mnist.shape)
print("Fashion-MNIST:", X_fashion.shape, y_fashion.shape)
print("Wine:", X_wine.shape, y_wine.shape)
print("Breast Cancer:", X_breast.shape, y_breast.shape)

# -----------------------------
# 2. Definir clasificadores
# -----------------------------
modelos = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, solver="lbfgs")),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma="scale")),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=80, max_depth=12, random_state=42, n_jobs=-1)
}

# -----------------------------
# 3. Función para evaluar un dataset
# -----------------------------
def evaluar_dataset(X, y, nombre):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    resultados = {}
    for modelo_nombre, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        resultados[modelo_nombre] = acc

        print(f"\n==== {nombre} - {modelo_nombre} ====")
        print("Accuracy:", round(acc, 4))
        print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred, zero_division=0))

    return resultados

# -----------------------------
# 4. Evaluar todos los datasets
# -----------------------------
resumen = {}
resumen["MNIST Digits"] = evaluar_dataset(X_mnist, y_mnist, "MNIST Digits")
resumen["Fashion-MNIST"] = evaluar_dataset(X_fashion, y_fashion, "Fashion-MNIST")
resumen["Wine"] = evaluar_dataset(X_wine, y_wine, "Wine")
resumen["Breast Cancer"] = evaluar_dataset(X_breast, y_breast, "Breast Cancer")

comparativa = pd.DataFrame(resumen).T.round(4)
print("\nCOMPARATIVA FINAL")
print(comparativa)

# -----------------------------
# 5. Visualización
# -----------------------------
plt.figure(figsize=(12, 6))
for i, (nombre, metricas) in enumerate(resumen.items(), 1):
    plt.subplot(2, 2, i)
    plt.bar(metricas.keys(), metricas.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    plt.title(nombre)
    plt.ylabel("Accuracy")
    plt.xticks(rotation=20)
    plt.ylim(0, 1.05)

plt.tight_layout()
plt.show()